# Generation of `SynthAddOnly` dataset used in CHEERS
###### Yacine MOKHTARI & Grégory SMITS - CHEERS

In [ ]:
from pathlib import Path
from itertools import product
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import pandas as pd
import random
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from scipy.sparse import coo_matrix
rng = np.random.default_rng()

#### i. Item catalog

In [ ]:
# Set of feature values
N_FEATURE_VALUES = 15

feature_values = {
    x: [x+str(i) for i in range(15)] for x in ['a', 'b', 'c']
}

# all combinations of feature values
items = list(product(*feature_values.values()))
print(f'three first items: {items[:3]}')

#### ii. User ground truth preferences
* We assume three groups of feature values per feature ([0-4, 5-9, 10-14])
* We create 4 types of users -- A, B, C, ABC (cf. article)

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

N_USERS = 100
FULL_REGIONS = True
rng = np.random.default_rng(42)

# Assumes items use the same feature order as feature_values.
features = list(feature_values.keys())

if set(features) != {"a", "b", "c"}:
    raise ValueError("Expected features named 'a', 'b', and 'c'.")
if N_USERS % 4 != 0:
    raise ValueError("N_USERS must be divisible by 4.")

fv_ids = [
    [0, 1, 2, 3, 4],
    [5, 6, 7, 8, 9],
    [10, 11, 12, 13, 14],
]
cluster_ids = np.arange(len(fv_ids))
number_interests = np.arange(1, len(fv_ids[0]) + 1)

# Used only when FULL_REGIONS=False.
alpha = 0.0  # alpha=0 gives uniform subset sizes.
probs = number_interests.astype(float) ** (-alpha)
probs /= probs.sum()

families = [
    ("a_only", ("a",)),
    ("b_only", ("b",)),
    ("c_only", ("c",)),
    ("abc", ("a", "b", "c")),
]
users_per_family = N_USERS // 4

# Inactive features have no preferred region or feature values.
user_interest = {
    u: {
        x: {"cluster_id": None, "feature_values": []}
        for x in features
    }
    for u in range(N_USERS)
}
user_family = {}

for family_idx, (family_name, active_features) in enumerate(families):
    start = family_idx * users_per_family
    stop = start + users_per_family

    for u in range(start, stop):
        user_family[u] = family_name

        for x in active_features:
            cluster = int(rng.choice(cluster_ids))

            if FULL_REGIONS:
                selected_ids = fv_ids[cluster]
            else:
                n = int(rng.choice(number_interests, p=probs))
                selected_ids = rng.choice(
                    fv_ids[cluster], size=n, replace=False
                )

            user_interest[u][x] = {
                "cluster_id": cluster,
                "feature_values": [
                    f"{x}{fv_id}" for fv_id in selected_ids
                ],
            }

# Ground-truth DataFrame.
rows = []

for u, user_features in user_interest.items():
    row = {"user_id": u, "family": user_family[u]}

    for x in features:
        row[f"{x}_cluster_id"] = user_features[x]["cluster_id"]
        row[f"{x}_feature_values"] = user_features[x]["feature_values"]

    rows.append(row)

cluster_columns = [f"{x}_cluster_id" for x in features]
value_columns = [f"{x}_feature_values" for x in features]

df = pd.DataFrame(
    rows,
    columns=["user_id", "family"] + cluster_columns + value_columns,
)

# Missing cluster IDs indicate inactive features.
df[cluster_columns] = df[cluster_columns].astype("Int64")

# Binary interactions: OR across the user's active features.
items_array = np.asarray(items)
liked = np.zeros((N_USERS, len(items)), dtype=bool)

for u in range(N_USERS):
    for col, x in enumerate(features):
        preferred_values = user_interest[u][x]["feature_values"]

        if preferred_values:
            liked[u] |= np.isin(
                items_array[:, col], preferred_values
            )

# Stores only positive interactions, all with value 1.
user_item_interactions = coo_matrix(liked, dtype=np.int8)

sparsity = 1.0 - (
    user_item_interactions.nnz / (N_USERS * len(items))
)

print(df["family"].value_counts())
print(f"Sparsity: {sparsity:.2%}")

#### iii. Interaction split -- random 80/20

In [ ]:
def split_per_user(matrix, train_ratio=0.8):
    matrix = matrix.tocsr(copy=True)
    matrix.sum_duplicates()
    matrix.eliminate_zeros()

    train_rows, train_cols, train_data = [], [], []
    test_rows, test_cols, test_data = [], [], []

    for user in range(matrix.shape[0]):
        start, end = matrix.indptr[user:user + 2]
        items = matrix.indices[start:end]
        values = matrix.data[start:end]

        order = rng.permutation(len(items))

        # Keep at least one training interaction for nonempty users.
        n_train = max(1, int(np.floor(train_ratio * len(items))))
        train_idx = order[:n_train]
        test_idx = order[n_train:]

        train_rows.extend([user] * len(train_idx))
        train_cols.extend(items[train_idx])
        train_data.extend(values[train_idx])

        test_rows.extend([user] * len(test_idx))
        test_cols.extend(items[test_idx])
        test_data.extend(values[test_idx])

    train = coo_matrix(
        (train_data, (train_rows, train_cols)),
        shape=matrix.shape,
        dtype=matrix.dtype,
    )
    test = coo_matrix(
        (test_data, (test_rows, test_cols)),
        shape=matrix.shape,
        dtype=matrix.dtype,
    )

    return train, test

train_interactions, test_interactions = split_per_user(user_item_interactions, 0.9)

In [ ]:
train_csr = train_interactions.tocsr()
test_csr = test_interactions.tocsr()

dict_ui = {} # {'user_id': [set(train_items), test_items, test_values]} --- input for CHEERS DataLoader

for user in range(train_csr.shape[0]):
    train_start, train_end = train_csr.indptr[user:user + 2]
    test_start, test_end = test_csr.indptr[user:user + 2]

    dict_ui[user] = [
        set(train_csr.indices[train_start:train_end].tolist()),
        test_csr.indices[test_start:test_end].copy(),
        test_csr.data[test_start:test_end].copy(),
    ]

assert len(dict_ui[0][0]) == sum(train_interactions.toarray()[0] != 0)

df_items_idx = pd.DataFrame(items, columns=['a', 'b', 'c'])
df_items_idx.insert(0, 'item_id', range(len(df_items_idx)))
for x in feature_values.keys():
    df_items_idx[x] = [elt[1:] for elt in df_items_idx[x]]

entity_mappings = {}
entity_mappings['item_to_idx'] = {i: i for i in range(len(df_items_idx))}
[entity_mappings.__setitem__(x, {i: v for i, v in enumerate(feature_values[x])}) for x in feature_values.keys()]

#### iv. Save everything for CHEERS DataLoader

In [ ]:
# # save as pkl - 
# import pickle
# import json

# save_path = './datasets/synthetic/RQ_region_recovery_study/'
# # create folder/path if doesnt exist
# import os
# if not os.path.exists(save_path):
#     os.makedirs(save_path)

# with open(save_path + 'df_item_idx.pkl', 'wb') as f:
#     pickle.dump(df_items_idx, f)
# with open(save_path + 'dict_ui.pkl', 'wb') as f:
#     pickle.dump(dict_ui, f)
# with open(save_path + 'user_item_matrix.pkl', 'wb') as f:
#     pickle.dump(train_interactions, f)
# with open(save_path + 'entity_mappings.pkl', 'wb') as f:
#     pickle.dump(entity_mappings, f)
# with open(save_path+'user_interest.json', 'w') as f:
#     json.dump(user_interest, f)


#### v. Example of loading

In [ ]:
# # save_path = './datasets/synthetic/RQ1_additive_regionsKnown/'
# save_path = './datasets/synthetic/RQ_region_recovery_study/'


# dataloader = Loader(path=save_path)
# vocab_sizes = dataloader.set_training_features(features=["a", "b", "c"])
# vocab_sizes 